# MSM sensitivity path

How wide are $P(Y=1\mid\mathrm{do}(X=1))$ bounds as Tan MSM $\Lambda$ goes from 1 → 5?


In [ ]:
from partialid import Assumptions, Query, Variables, bound, bound_path

V = Variables.from_binary("X", "Y")
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)


## Point estimate at a fixed budget


In [ ]:
r = bound(V, P, Q, Assumptions.sensitivity_msm("X", "Y", Lambda=3.0))
print(r.summary(Q))
assert abs(r.width - 1 / 3) < 1e-3


## Sensitivity path in $\Lambda$

At $\Lambda=1$ the effect is point-identified on this margin. Larger $\Lambda$ widens bounds.


In [ ]:
path = bound_path(
    V, P, Q, Assumptions.manski("X", "Y"),
    param="Lambda", theta_min=1.0, theta_max=5.0,
)
print(path.summary(at=3.0))
print(f"closed_form={path.closed_form}")
for lam in (1.0, 2.0, 3.0, 5.0):
    lo, hi, w = path.at(lam)
    print(f"Λ={lam:g}: [{lo:.4f}, {hi:.4f}]  width={w:.4f}")


## Breakdown point — when does a null enter the interval?


In [ ]:
lam_star = path.breakdown(null=0.5)
print(f"breakdown(null=0.5) → Λ* = {lam_star}")


## Same path with MTR (stricter)


In [ ]:
path_mtr = bound_path(
    V, P, Q,
    Assumptions.manski("X", "Y") & Assumptions.mtr("Y", "X"),
    param="Lambda", theta_min=1.0, theta_max=5.0,
)
for lam in (1.0, 2.0, 3.0):
    lo, hi, w = path_mtr.at(lam)
    lo0, hi0, w0 = path.at(lam)
    print(f"Λ={lam:g}: Manski w={w0:.4f} → +MTR w={w:.4f}")
